# The website, on Colab's GPU

Runs `app/app.py` on a Colab GPU and gives you a public link. Open the link in a new tab, type a prompt, drag the switch slider, press **Generate**. Keep this tab open while you use it; closing it stops the server.

Setup: Runtime > Change runtime type > **T4 GPU** > Save. Then Runtime > Run all. The first Generate click loads the models (~10 GB download, 3-5 minutes); later clicks take about 20-40 seconds per image on a T4.

In [1]:
# --- Setup (Colab: Runtime > Change runtime type > T4 GPU or better) ---
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""   # paste your GitHub URL here once the repo is pushed; leave empty to upload the zip instead
if IN_COLAB and not os.path.exists("image-rerouter"):
    if REPO_URL:
        !git clone -q $REPO_URL
    else:
        from google.colab import files
        import glob, shutil
        print("Pick the image-rerouter zip (any name, e.g. image-rerouter_2026-10-26.zip) in the file dialog below.")
        ZIP = next(iter(files.upload()))
        !rm -rf /content/_unzip && unzip -q -o "$ZIP" -d /content/_unzip && rm "$ZIP"
        # the code is wherever config.yaml sits next to rerouter/, whatever the zip's folder is called
        src = next(os.path.dirname(c) for c in glob.glob("/content/_unzip/**/config.yaml", recursive=True)
                   if os.path.isdir(os.path.join(os.path.dirname(c), "rerouter")))
        shutil.move(src, "/content/image-rerouter")
if IN_COLAB:
    %cd image-rerouter
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao   # Colab's preinstalled torchao is too old for new diffusers (FqnToConfig import error); unused here
# optional: faster downloads + higher rate limits
# from huggingface_hub import login; login()
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
from rerouter import utils
cfg = utils.load_config()
cfg

Pick the image-rerouter zip (any name, e.g. image-rerouter_2026-10-26.zip) in the file dialog below.


Saving image-rerouter_2026-11-23.zip to image-rerouter_2026-11-23.zip
/content/image-rerouter
GPU: Tesla T4


{'models': {'large': 'stabilityai/stable-diffusion-xl-base-1.0',
  'small': 'segmind/SSD-1B',
  'tiny': 'segmind/Segmind-Vega',
  'vae': 'madebyollin/sdxl-vae-fp16-fix',
  'text_encoders_from': 'large'},
 'generation': {'steps': 30,
  'width': 1024,
  'height': 1024,
  'guidance_scale': 7.5,
  'scheduler': 'EulerDiscreteScheduler',
  'negative_prompt': '',
  'dtype': 'float16'},
 'seeds': [42, 1234],
 'switch_fractions': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
 'control_step_counts': [12, 18, 24, 30],
 'handoff': {'impl': 'swap', 'warmup_runs': 1},
 'compute': {'gpu_type': 'T4',
  'usd_per_hour': {'Tesla T4': 0.35, 'dry-run': 0.35},
  'attention_slicing': False,
  'vae_slicing': True,
  'step_seconds': {'Tesla T4': {'large': 0.81, 'small': 0.517, 'fixed': 0.87},
   'NVIDIA RTX PRO 6000 Blackwell Server Edition': {'large': 0.05898,
    'small': 0.03811,
    'fixed': 0.1255},
   'dry-run': {'large': 0.8, 'small': 0.45, 'tiny': 0.3, 'fixed': 0.9}}},
 'scoring': {'pick_score': True,
  'pickscore_m

In [ ]:
APP_PASSWORD = ""   # set one so only people you give it to can use your GPU (user name: guest); blank = no login
os.environ["RR_AUTH"] = f"guest:{APP_PASSWORD}" if APP_PASSWORD else ""
!python app/app.py --share

Models load on the first Generate click.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://04086b2a6e07ccc6af.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
config.json: 100% 631/631 [00:00<00:00, 2.35MB/s]

diffusion_pytorch_model.safetensors: downloading bytes:   0% 0.00/335M [00:00<?, ?B/s]
diffusion_pytorch_model.safetensors: downloading bytes:  11% 36.0M/335M [00:00<00:04, 66.2MB/s,  502kB/s  ]
diffusion_pytorch_model.safetensors: downloading bytes:  18% 61.6M/335M [00:00<00:02, 106MB